In [1]:
from transformers import BertTokenizer, BertForMaskedLM, Trainer, TrainingArguments, GPT2LMHeadModel


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

In [2]:
import numpy as np

with open("dataset/sherlock_holmes.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])

Sherlock Holmes, the iconic fictional detective created by Sir Arthur Conan Doyle, is a character that has captivated audiences since his first appearance in 1887's *A Study in Scarlet*. Known for his


In [3]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

model = GPTNeoForCausalLM.from_pretrained('EleutherAI/gpt-neo-125M')
tokenizer = GPT2Tokenizer.from_pretrained('EleutherAI/gpt-neo-125M')

In [4]:
from datasets import Dataset

In [5]:
dataset = Dataset.from_text("dataset/sherlock_holmes.txt")

In [6]:
dataset

Dataset({
    features: ['text'],
    num_rows: 35
})

In [7]:
tokenizer.pad_token = tokenizer.eos_token
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=28)

In [8]:
train_test_split = dataset.train_test_split(test_size=0.1)  # Split 10% for validation

train_dataset = train_test_split['train']
validation_dataset = train_test_split['test']

# Verify the sizes of the splits
print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['text'],
    num_rows: 31
})
Dataset({
    features: ['text'],
    num_rows: 4
})


In [9]:
validation_test_split = validation_dataset.train_test_split(test_size=0.5)

validation_dataset = validation_test_split['train']
test_dataset = validation_test_split['test']

# Verify the sizes of the splits
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text'],
    num_rows: 2
})
Dataset({
    features: ['text'],
    num_rows: 2
})


In [10]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [11]:
def add_labels(example):
    # For masked language modeling, labels are the same as input_ids, but we mask some tokens
    example['labels'] = example['input_ids'].copy()  # Copy input_ids to labels
    
    # You may want to add masking logic here if you're doing masked language modeling
    # For simplicity, we're not applying any masking here
    return example

In [12]:
train_dataset = train_dataset.map(add_labels, batched=True)
train_dataset

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 31
})

In [13]:
validation_dataset = validation_dataset.map(add_labels, batched=True)
validation_dataset

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

In [14]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    remove_unused_columns=False,  # Prevent automatic removal of unused columns
)

# Setup the Trainer with the new splits
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # Use the new train set
    eval_dataset=validation_dataset,  # Use the new validation set
)

In [15]:
trainer.train() # going for 7 hours?

  0%|          | 0/80 [00:00<?, ?it/s]

{'train_runtime': 658.0298, 'train_samples_per_second': 0.942, 'train_steps_per_second': 0.122, 'train_loss': 1.3505075454711915, 'epoch': 20.0}


TrainOutput(global_step=80, training_loss=1.3505075454711915, metrics={'train_runtime': 658.0298, 'train_samples_per_second': 0.942, 'train_steps_per_second': 0.122, 'total_flos': 8856553144320.0, 'train_loss': 1.3505075454711915, 'epoch': 20.0})

In [16]:
# After training, you can save the model to a directory
output_dir = "gpt_neo/"

# Saving the model and tokenizer
trainer.save_model(output_dir)  # Will save the model weights and the tokenizer
tokenizer.save_pretrained(output_dir)


('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/vocab.json',
 'model/merges.txt',
 'model/added_tokens.json')

In [17]:
import torch
device = torch.device("cpu")
model.to(device)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [18]:
input_text = "sherlock homes the iconic"
input_ids = tokenizer(input_text, return_tensors="pt")
input_ids["attention_mask"] = (input_ids["input_ids"] != tokenizer.pad_token_id).long()

In [19]:
input_ids = input_ids.to(device)
input_ids

{'input_ids': tensor([[   82,   372,  5354,  5682,   262, 14133]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [21]:
# Generate continuation
outputs = model.generate(input_ids.input_ids, max_length=20, pad_token_id=tokenizer.pad_token_id)

# Decode the generated tokens
generated_text = tokenizer.decode(outputs[0])
print("\ngenerated_text : ",generated_text)


generated_text :  sherlock homes the iconic character of the character, the character that is the most iconic of the


In [22]:
input_text = "sherlock homes the iconic"
input_ids = tokenizer(input_text, return_tensors="pt")
input_ids["attention_mask"] = (input_ids["input_ids"] != tokenizer.pad_token_id).long()

outputs = model.generate(input_ids.input_ids, max_length=20, pad_token_id=tokenizer.pad_token_id)

# Decode the generated tokens
generated_text = tokenizer.decode(outputs[0])
print("\ngenerated_text : ",generated_text)


generated_text :  sherlock homes the iconic character of the character, the character that is the most iconic of the


In [25]:
input_text = "The stories of Sherlock Holmes"
input_ids = tokenizer(input_text, return_tensors="pt")
input_ids["attention_mask"] = (input_ids["input_ids"] != tokenizer.pad_token_id).long()

outputs = model.generate(input_ids.input_ids, max_length=40, pad_token_id=tokenizer.pad_token_id)

# Decode the generated tokens
generated_text = tokenizer.decode(outputs[0])
print("\ngenerated_text : ",generated_text)


generated_text :  The stories of Sherlock Holmes have had a lasting impact on the detective genre. His character's unique ability to solve complex cases has made him one of the most popular detectives in the world. His character's
